In [ ]:
import psycopg2
import pandas as pd

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    try:
        connection = psycopg2.connect(
            host=host_ip, database=database_name, user=user, password=password, port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")
        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None
    finally:
        if 'connection' in locals():
            connection.close()

# --- Configuration ---
HOST_IP = "100.94.14.115"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

In [54]:
df = df_original.copy(deep=True)
df = df[
    (df['dept_name'] == 'Rolling-Stock') & (df['status_id'] == 1)
][['filename', 'workorder_id', 'interval', 'dept_name', 'json_data']]

df

,filename,workorder_id,interval,dept_name,json_data
0,RS_PM_WEK_4000586856.pdf,4.000587e+09,Weekly,Rolling-Stock,{'notification': {'notification_no': '12244350...
1,RS_PM_MTH_4000464193.pdf,4.000464e+09,Monthly,Rolling-Stock,{'notification': {'notification_no': '11945789...
3,RS_PM_MTH_4000446287.pdf,4.000446e+09,Monthly,Rolling-Stock,{'notification': {'notification_no': '11898977...
4,RS_PM_WEK_4000558454.pdf,4.000558e+09,Weekly,Rolling-Stock,{'notification': {'notification_no': '12179199...
5,RS_PM_MTH_4000464732.pdf,4.000465e+09,Monthly,Rolling-Stock,{'notification': {'notification_no': '11946842...
...,...,...,...,...,...
7590,RS_PM_WEK_4000538576.pdf,4.000539e+09,Weekly,Rolling-Stock,{'notification': {'notification_no': '12129789...
7591,RS_PM_WEK_4000491866.pdf,4.000492e+09,Weekly,Rolling-Stock,{'notification': {'notification_no': '12013423...
7592,RS_PM_YRL_4000457606.pdf,4.000458e+09,Yearly,Rolling-Stock,{'notification': {'notification_no': '11926372...
7593,RS_PM_WEK_4000658422.pdf,4.000658e+09,Weekly,Rolling-Stock,{'notification': {'notification_no': '12410787...


In [55]:
import json

def normalize(x):
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        try:
            return json.loads(x)
        except:
            return None
    return None

df["json_data"] = df["json_data"].apply(normalize)

In [56]:
def safe_get(d, keys):
    for k in keys:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
    return d

In [57]:
import pandas as pd

extracted_df = pd.DataFrame({
    
    "workorder_no": df["workorder_id"],

    "plan_start_date_time": df["json_data"].apply(
        lambda x: x.get("notification", {})
                   .get("staff_details", [{}])[0]
                   .get("plan_start_date_time")
    ),

    "plan_end_date_time": df["json_data"].apply(
        lambda x: x.get("notification", {})
                   .get("staff_details", [{}])[0]
                   .get("plan_end_date_time")
    ),

    "comments": df["json_data"].apply(
        lambda x: x.get("notification", {})
                   .get("comments")
    ),

    "supervisor": df["json_data"].apply(
        lambda x: (
            ((((x or {}).get("stamping") or {})
                .get("power_off") or {})
                .get("inspections") or {})
                .get("preparation") or {}
        )
        .get("signed", {})
        .get("supervisor")
    ),
    
    # POWER OFF

    "Poff_ECA1": df["json_data"].apply(
        lambda x: safe_get(x, [
            "stamping",
            "power_off",
            "inspections",
            "preparation",
            "checks",
            "ECA1"
        ])
    ),

    "Poff_ICA2": df["json_data"].apply(
        lambda x: safe_get(x, [
            "stamping",
            "power_off",
            "inspections",
            "preparation",
            "checks",
            "ICA2"
        ])
    ),
    
    "Poff_ICA3": df["json_data"].apply(
        lambda x: safe_get(x, [
            "stamping",
            "power_off",
            "inspections",
            "preparation",
            "checks",
            "ICA3"
        ])
    ),
    
    "Poff_ECA4": df["json_data"].apply(
        lambda x: safe_get(x, [
            "stamping",
            "power_off",
            "inspections",
            "preparation",
            "checks",
            "ECA4"
        ])
    ),
    
    # POWER ON
    "Pon_ECA1": df["json_data"].apply(
        lambda x: safe_get(x, [
            "stamping",
            "power_on",
            "inspections",
            "preparation",
            "checks",
            "ECA1"
        ])
    ),

    "Pon_ICA2": df["json_data"].apply(
        lambda x: safe_get(x, [
            "stamping",
            "power_on",
            "inspections",
            "preparation",
            "checks",
            "ICA2"
        ])
    ),
    
    "Pon_ICA3": df["json_data"].apply(
        lambda x: safe_get(x, [
            "stamping",
            "power_on",
            "inspections",
            "preparation",
            "checks",
            "ICA3"
        ])
    ),
    
    "Pon_ECA4": df["json_data"].apply(
        lambda x: safe_get(x, [
            "stamping",
            "power_on",
            "inspections",
            "preparation",
            "checks",
            "ECA4"
        ])
    ),
    
    "filename": df["filename"]

})

extracted_df.head()

,workorder_no,plan_start_date_time,plan_end_date_time,comments,supervisor,Poff_ECA1,Poff_ICA2,Poff_ICA3,Poff_ECA4,Pon_ECA1,Pon_ICA2,Pon_ICA3,Pon_ECA4,filename
0,4.000587e+09,25/02/2024 22:12:00,26/02/24 03:30:00,NA,7127,"19580,7202,11515,10475,20674","19580,7202,11515,10475,20674","19580,7202,11515,10475,20674","19580,7202,11515,10475,20674","20674,19580,11515,7202,10475","20674,19580,11515,7202,10475","20674,19580,11515,7202,10475","20674,19580,11515,7202,10475",RS_PM_WEK_4000586856.pdf
1,4.000464e+09,"10/05/2022, 22:00:00","13/05/2022, 04:00:00",,7066,"7203, 7232, 7270, 1054, 7306","7203, 7232, 7270, 1054, 7306","7203, 7232, 7270, 1054, 7306","7203, 7232, 7270, 1054, 7306","7203, 7232, 7270, 1054, 7306","7203, 7232, 7270, 1054, 7306","7203, 7232, 7270, 1054, 7306","7203, 7232, 7270, 1054, 7306",RS_PM_MTH_4000464193.pdf
3,4.000446e+09,25/01/2022 22:10:00,26/01/2022 03:56:00,"Initial time: 2210, Initial Date: 25/01/22, Fi...",7127,"11139,7205,7203,7270,9031","11139,7205,7203,7270,9031","11139,7205,7203,7270,9031","11139,7205,7203,7270,9031","7205,11139,9031,7203,7270","7205,11139,9031,7203,7270","7205,11139,9031,7203,7270","7205,11139,9031,7203,7270",RS_PM_MTH_4000446287.pdf
4,4.000558e+09,"09/10/2023, 10:00:00","10/10/2023, 04:30:00",NA,7192,19791,19791,19791,19791,7205,12487,11083,19791,RS_PM_WEK_4000558454.pdf
5,4.000465e+09,"17/05/2022, 22:20:00","18/05/2022, 04:10:00",,7127,"7203, 9031, 7270,7205","7203, 9031, 7270,7205","7203, 9031, 7270,7205","7203, 9031, 7270,7205","7205, 7276, 7203, 9031","7205, 7276, 7203, 9031","7205, 7276, 7203, 9031","7205, 7276, 7203, 9031",RS_PM_MTH_4000464732.pdf


In [58]:
extracted_df.to_excel("extracted/extracted_rsd.xlsx", index=False)

In [59]:
import pandas as pd

df_rsd = pd.read_excel("extracted/extracted_rsd.xlsx")

cols = ["Poff_ECA1", "Poff_ICA2", "Poff_ICA3", "Poff_ECA4", "Pon_ECA1", "Pon_ICA2", "Pon_ICA3", "Pon_ECA4"]

for col in cols:
    df_rsd[col] = df_rsd[col].astype(str).str.replace(r"[\[\]]", "", regex=True)
    df_rsd[col] = df_rsd[col].astype(str).str.replace(r"'", "", regex=True)
    df_rsd[col] = df_rsd[col].astype(str).str.replace(r" ", "", regex=True)

df_rsd["checks_values"] = df_rsd[cols].astype(str).agg(",".join, axis=1)


### THIS IS NEW DF FOR UNIQUE CHECKS with filename and workorder_no preset
cleaned_df = df_rsd[["filename", "workorder_no", "plan_start_date_time", "plan_end_date_time", "comments", "supervisor", "checks_values"]].copy()

cleaned_df["technician_ids"] = cleaned_df["checks_values"].apply(
    lambda x: ",".join(
        dict.fromkeys([i.strip() for i in x.split(",") if i.strip()])
    ) if isinstance(x, str) else None
)

cleaned_df.drop("checks_values", axis=1).head()

,filename,workorder_no,plan_start_date_time,plan_end_date_time,comments,supervisor,technician_ids
0,RS_PM_WEK_4000586856.pdf,4.000587e+09,25/02/2024 22:12:00,26/02/24 03:30:00,NaN,7127,"19580,7202,11515,10475,20674"
1,RS_PM_MTH_4000464193.pdf,4.000464e+09,"10/05/2022, 22:00:00","13/05/2022, 04:00:00",NaN,7066,"7203,7232,7270,1054,7306"
2,RS_PM_MTH_4000446287.pdf,4.000446e+09,25/01/2022 22:10:00,26/01/2022 03:56:00,"Initial time: 2210, Initial Date: 25/01/22, Fi...",7127,"11139,7205,7203,7270,9031"
3,RS_PM_WEK_4000558454.pdf,4.000558e+09,"09/10/2023, 10:00:00","10/10/2023, 04:30:00",NaN,7192,"19791,7205,12487,11083"
4,RS_PM_MTH_4000464732.pdf,4.000465e+09,"17/05/2022, 22:20:00","18/05/2022, 04:10:00",NaN,7127,"7203,9031,7270,7205,7276"


### Comparing list of technician and supervisor with tbl_user

In [60]:
ref_user = pd.read_excel("tbl_users.xlsx")
ref_user.head()

,id,staff_id,name,call_sign,position,department,email
0,1,10018972,LUQMAN NULHAKIM BIN JAMALUDIN,LNJ 18972,Senior Associate,Power System,luqnman.jamaludin@prasarana.com.my
1,2,10007223,ROHAIZAN BIN MASTOR,RM7223,Associate,Power System,rohaizan@prasarana.com.my
2,3,10023969,MUHAMMAD NUR ZAM ZAM BIN MOHD ROSLE,NZZ23969,Associate,Power System,nurzamzam.rosle@prasarana.com.my
3,4,10025298,IDHAR DANIEL BIN MOHD AZHAR,IDA25298,Associate,Power System,idhar.azhar@prasarana.com.my
4,5,10024324,MUHAMMAD IKHWAN BIN ABDULLAH,MIA24324,Senior Associate,Power System,ikhwan.abdullah@prasarana.com.my


In [61]:
cleaned_user = ref_user[["staff_id", "name", "call_sign", "department"]].copy()

cleaned_user["stamp_id"] = (
    cleaned_user["staff_id"]
    .astype(str)
    .str.replace(r"^1000|^100", "", regex=True)
)

cleaned_user["stamp_id"] = cleaned_user["stamp_id"].astype(str)
cleaned_user.head()

,staff_id,name,call_sign,department,stamp_id
0,10018972,LUQMAN NULHAKIM BIN JAMALUDIN,LNJ 18972,Power System,18972
1,10007223,ROHAIZAN BIN MASTOR,RM7223,Power System,7223
2,10023969,MUHAMMAD NUR ZAM ZAM BIN MOHD ROSLE,NZZ23969,Power System,23969
3,10025298,IDHAR DANIEL BIN MOHD AZHAR,IDA25298,Power System,25298
4,10024324,MUHAMMAD IKHWAN BIN ABDULLAH,MIA24324,Power System,24324


In [62]:
id_to_name = dict(zip(
    cleaned_user["stamp_id"].astype(str),
    cleaned_user["name"]
))

id_to_supervisor_name = id_to_name

def build_rows(row):
    
    tech_ids = row["technician_ids"]

    if not isinstance(tech_ids, str):
        return []

    tech_list = [i.strip() for i in tech_ids.split(",") if i.strip()]

    results = []

    for tech_id in tech_list:
        
        results.append({
            "filename": row.get("filename"),
            "workorder_no": row.get("workorder_no"),
            "plan_start_date_time": row.get("plan_start_date_time"),
            "plan_end_date_time": row.get("plan_end_date_time"),
            "comments": row.get("comments"),

            "technician_id": tech_id,
            "name": id_to_name.get(tech_id, ""),

            "supervisor_id": row.get("supervisor"),
            "supervisor_name": id_to_supervisor_name.get(
                str(row.get("supervisor")), ""
            ),
        })

    return results

expanded = cleaned_df.apply(build_rows, axis=1).explode().dropna()

final_df = pd.DataFrame(expanded.tolist())
final_df.head()

,filename,workorder_no,plan_start_date_time,plan_end_date_time,comments,technician_id,name,supervisor_id,supervisor_name
0,RS_PM_WEK_4000586856.pdf,4.000587e+09,25/02/2024 22:12:00,26/02/24 03:30:00,NaN,19580,MUHAMMAD DANIEL BIN ZAMRI,7127,MOHD FAHRIS BIN NORDIN
1,RS_PM_WEK_4000586856.pdf,4.000587e+09,25/02/2024 22:12:00,26/02/24 03:30:00,NaN,7202,ABUZAR BIN HASHIM @SHAHAR,7127,MOHD FAHRIS BIN NORDIN
2,RS_PM_WEK_4000586856.pdf,4.000587e+09,25/02/2024 22:12:00,26/02/24 03:30:00,NaN,11515,MOHAMAD FARIS BIN FAUZI,7127,MOHD FAHRIS BIN NORDIN
3,RS_PM_WEK_4000586856.pdf,4.000587e+09,25/02/2024 22:12:00,26/02/24 03:30:00,NaN,10475,MUHAMMAD DZULIZHAM BIN MAZLAN,7127,MOHD FAHRIS BIN NORDIN
4,RS_PM_WEK_4000586856.pdf,4.000587e+09,25/02/2024 22:12:00,26/02/24 03:30:00,NaN,20674,MUHAMAD AQIL HARIS BIN MAT AKHIR,7127,MOHD FAHRIS BIN NORDIN


In [63]:
final_df.to_excel("output/staff_rsd.xlsx", index=False)